In [ ]:
!pip install nltk
import nltk

In [ ]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('brown')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!


True

In [ ]:
from nltk.corpus import brown
from nltk.util import ngrams
from collections import Counter

In [ ]:
# Load sentences from the brown 'news' category
sentences = brown.sents(categories='news')

# show fisrt few sentences
sentences[:5]


[['The',
  'Fulton',
  'County',
  'Grand',
  'Jury',
  'said',
  'Friday',
  'an',
  'investigation',
  'of',
  "Atlanta's",
  'recent',
  'primary',
  'election',
  'produced',
  '``',
  'no',
  'evidence',
  "''",
  'that',
  'any',
  'irregularities',
  'took',
  'place',
  '.'],
 ['The',
  'jury',
  'further',
  'said',
  'in',
  'term-end',
  'presentments',
  'that',
  'the',
  'City',
  'Executive',
  'Committee',
  ',',
  'which',
  'had',
  'over-all',
  'charge',
  'of',
  'the',
  'election',
  ',',
  '``',
  'deserves',
  'the',
  'praise',
  'and',
  'thanks',
  'of',
  'the',
  'City',
  'of',
  'Atlanta',
  "''",
  'for',
  'the',
  'manner',
  'in',
  'which',
  'the',
  'election',
  'was',
  'conducted',
  '.'],
 ['The',
  'September-October',
  'term',
  'jury',
  'had',
  'been',
  'charged',
  'by',
  'Fulton',
  'Superior',
  'Court',
  'Judge',
  'Durwood',
  'Pye',
  'to',
  'investigate',
  'reports',
  'of',
  'possible',
  '``',
  'irregularities',
  "''",
 

In [ ]:
# Flatten the corpus into a sngle list of words
corpus = [w.lower() for sent in sentences for w in sent]

# Unigrams counts
unigram_counts = Counter(corpus)

# Bigram counts
bigram_counts = Counter()
for sent in sentences:
  sent = [w.lower() for w in sent]
  bigram_counts.update(ngrams(sent, 2))

# Trigram counts
trigram_counts = Counter()
for sent in sentences:
  sent = [w.lower() for w in sent]
  trigram_counts.update(ngrams(sent, 3))

In [ ]:
total_words = len(corpus)

# Unigram Probabilities
unigram_probs = {w: c/total_words for w, c in unigram_counts.items()}

# Bigram Probabilities
bigram_probs = {
    (w1, w2): c / unigram_counts[w1]
    for (w1, w2), c in bigram_counts.items()
}

# Bigram Probabilities
trigram_probs = {
    (w1, w2, w3): c / bigram_counts[(w1, w2)]
    for (w1, w2, w3), c in trigram_counts.items()
}

In [ ]:
print('=== Sample Unigram Probabilities ===')
for w in list(unigram_probs.keys())[:10]:
  print(f'P({w}) = {unigram_probs[w]:.6f}')

print('\n=== Sample Bigram Probabilities ===')
for bg in list(bigram_probs.keys())[:10]:
  print(f'P({bg[1]} | {bg[0]}) = {bigram_probs[bg]:.6f}')

print('\n=== Sample Trigram Probabilities ===')
for tg in list(trigram_probs.keys())[:10]:
  print(f'P({tg[2]} | {tg[0]}, {tg[1]}) = {trigram_probs[tg]:.6f}')

=== Sample Unigram Probabilities ===
P(the) = 0.063508
P(fulton) = 0.000139
P(county) = 0.000607
P(grand) = 0.000189
P(jury) = 0.000457
P(said) = 0.004038
P(friday) = 0.000408
P(an) = 0.003093
P(investigation) = 0.000109
P(of) = 0.028452

=== Sample Bigram Probabilities ===
P(fulton | the) = 0.000940
P(county | fulton) = 0.428571
P(grand | county) = 0.016393
P(jury | grand) = 0.421053
P(said | jury) = 0.173913
P(friday | said) = 0.009852
P(an | friday) = 0.024390
P(investigation | an) = 0.012862
P(of | investigation) = 0.363636
P(atlanta's | of) = 0.000350

=== Sample Trigram Probabilities ===
P(county | the, fulton) = 0.500000
P(grand | fulton, county) = 0.166667
P(jury | county, grand) = 1.000000
P(said | grand, jury) = 0.125000
P(friday | jury, said) = 0.125000
P(an | said, friday) = 0.250000
P(investigation | friday, an) = 1.000000
P(of | an, investigation) = 0.750000
P(atlanta's | investigation, of) = 0.250000
P(recent | of, atlanta's) = 1.000000


In [ ]:
def sentence_prob(sentence, model="unigram"):
  import nltk
  words = nltk.word_tokenize(sentence.lower())
  prob = 1.0

  if model == "unigram":
    for w in words:
      prob *= unigram_probs.get(w, 1e-9)

  elif model == "bigram":
    for i in range(1, len(words)):
      pair = (words[i-1], words[i])
      prob *= bigram_probs.get(pair, 1e-9)

  elif model == "trigram":
    for i in range(2, len(words)):
      triple = (words[i-2], words[i-1], words[i])
      prob *= trigram_probs.get(triple, 1e-9)

  return prob

In [ ]:
test_sentence = "the stock market fell sharply today"
print("Sentence: ", test_sentence)
print("Unigram Probability: ", sentence_prob(test_sentence, "unigram"))
print("Bigram Probability: ", sentence_prob(test_sentence, "bigram"))
print("Trigram Probability: ", sentence_prob(test_sentence, "trigram"))

Sentence:  the stock market fell sharply today
Unigram Probability:  0.0005469697873779263
Bigram Probability:  1.043950307965341e-31
Trigram Probability:  1.0000000000000003e-36
